# Data Quality & Cleaning

This notebook audits the raw credit applications dataset for data quality issues across five dimensions: **Completeness**, **Consistency**, **Validity**, **Uniqueness**, and **Accuracy**. Each section documents findings, quantifies the extent of each issue, and applies remediation steps to produce a clean dataset for downstream bias analysis.

In [94]:
import os
import re
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Table of Contents

1. [Initial Data Inspection](#1-initial-data-inspection)
   - 1.1 [Load & Preview](#11-load--preview)
   - 1.2 [Dataset Shape](#12-dataset-shape)
   - 1.3 [Outcome Distribution](#13-outcome-distribution)
   - 1.4 [Data Types Overview](#14-data-types-overview)
   - 1.5 [Spending Behavior](#15-spending-behavior)
   - 1.6 [Findings](#16-findings)
2. [Assessment Structure](#2-assessment-structure)
3. [Completeness](#3-completeness)
   - 3.1 [Missing Values per Column](#31-missing-values-per-column)
   - 3.2 [annual_income vs annual_salary](#32-annual_income-vs-annual_salary)
   - 3.3 [Spending Behavior Missingness](#33-spending-behavior-missingness)
   - 3.4 [Findings](#34-findings)
4. [Consistency](#4-consistency)
   - 4.1 [Numeric Data Types](#41-numeric-data-types)
   - 4.2 [Gender Categorical Consistency](#42-gender-categorical-consistency)
   - 4.3 [Date Format Consistency](#43-date-format-consistency)
      - 4.3.1 [date_of_birth formats](#431-date_of_birth-formats)
      - 4.3.2 [processing_timestamp](#432-processing_timestamp)
      - 4.3.3 [Derived Feature: Age](#433-derived-feature-age)
   - 4.4 [Findings](#44-findings)
5. [Validity](#5-validity)
   - 5.1 [Negative Numeric Values](#51-negative-numeric-values)
   - 5.2 [SSN Format](#52-ssn-format)
   - 5.3 [ZIP Code Format](#53-zip-code-format)
   - 5.4 [Email Format](#54-email-format)
   - 5.5 [Findings](#55-findings)
6. [Uniqueness](#6-uniqueness)
7. [Accuracy](#7-accuracy)
   - 7.1 [Plausible Ages](#71-plausible-ages)
   - 7.2 [Approved Loans with Zero or Missing Income](#72-approved-loans-with-zero-or-missing-income)
   - 7.3 [Findings](#73-findings)
8. [Summary of Data Quality Issues](#8-summary-of-data-quality-issues)

## 1. Initial Data Inspection

We inspect the dataset structure, number of records, and variable types to identify potential data quality issues before any cleaning is applied.

### 1.1 Load & Preview

We load the raw JSON file and flatten nested fields using *pd.json_normalize*.

In [95]:
with open('../data/raw_credit_applications.json') as f:
    data = json.load(f)

df = pd.json_normalize(data)
df.head()

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,...,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,...,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,...,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,...,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,...,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN


### 1.2 Dataset Shape

In [96]:
print(f'Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns')

Dataset shape: 502 rows x 21 columns


### 1.3 Outcome Distribution

We check how many applications were approved versus rejected to understand the class balance.

In [97]:
approval_counts = df['decision.loan_approved'].value_counts()
approval_rate = df['decision.loan_approved'].mean() * 100
print('Loan approval counts:\n', approval_counts.to_string())
print(f'\nOverall approval rate: {approval_rate:.1f}%')

Loan approval counts:
 decision.loan_approved
True     292
False    210

Overall approval rate: 58.2%


### 1.4 Data Types Overview

We print the inferred data types for each column after JSON flattening.

In [98]:
print(df.dtypes.to_string())

_id                                  object
spending_behavior                    object
processing_timestamp                 object
applicant_info.full_name             object
applicant_info.email                 object
applicant_info.ssn                   object
applicant_info.ip_address            object
applicant_info.gender                object
applicant_info.date_of_birth         object
applicant_info.zip_code              object
financials.annual_income             object
financials.credit_history_months      int64
financials.debt_to_income           float64
financials.savings_balance            int64
decision.loan_approved                 bool
decision.rejection_reason            object
loan_purpose                         object
decision.interest_rate              float64
decision.approved_amount            float64
financials.annual_salary            float64
notes                                object


Some columns appear to have unexpected types — for example, *financials.annual_income* is stored as *object* because some records contain string-formatted numbers. This will be investigated and corrected in the Consistency section.

### 1.5 Spending Behavior

The *spending_behavior* column contains a nested list of *{category, amount}* objects per applicant. We expand it into a flat dataframe to understand the spending categories and amounts present.

In [99]:
# Expand spending_behavior into a flat dataframe
spending_rows = []
for idx, row in df.iterrows():
    sb = row['spending_behavior']
    if isinstance(sb, list):
        for entry in sb:
            spending_rows.append({'app_id': row['_id'], 'category': entry.get('category'), 'amount': entry.get('amount')})

df_spending = pd.DataFrame(spending_rows)
print('Spending behavior records:', len(df_spending))
print('\nCategories found:')
print(df_spending['category'].value_counts().to_string())
print('\nAmount stats per category:')
print(df_spending.groupby('category')['amount'].describe().round(1).to_string())

Spending behavior records: 827

Categories found:
category
Travel                 80
Utilities              76
Entertainment          72
Fitness                71
Insurance              68
Healthcare             68
Dining                 66
Groceries              65
Education              64
Transportation         61
Rent                   59
Shopping               54
Alcohol                11
Gambling                7
Adult Entertainment     5

Amount stats per category:
                     count   mean    std    min    25%    50%    75%    max
category                                                                   
Adult Entertainment    5.0  591.2  201.0  322.0  497.0  579.0  710.0  848.0
Alcohol               11.0  477.1  221.2  194.0  247.0  447.0  672.0  757.0
Dining                66.0  472.8  237.2   75.0  266.0  462.0  650.2  897.0
Education             64.0  499.7  255.8   54.0  268.8  528.0  713.8  889.0
Entertainment         72.0  496.0  240.6   52.0  291.2  489.0  715.

Spending behavior is available for all applicants but is not used as a predictor in this analysis, as spending category data could act as a socioeconomic proxy for protected characteristics such as race or gender — raising fairness concerns under GDPR and the EU AI Act. We document it here for completeness.

### 1.6 Findings

**Findings:**
- The dataset contains 502 rows and 21 columns after JSON flattening.
- The overall approval rate is approximately 58%, indicating a moderate class imbalance.
- Several columns have unexpected data types (e.g., *financials.annual_income* stored as *object*), which will be corrected in Section 4.
- Spending behavior is fully populated for all records and spans multiple categories including Rent, Shopping, Fitness, and Entertainment.
- The spending behavior data includes sensitive categories such as Alcohol, Gambling, and Adult Entertainment, which are ethically significant from a governance perspective and are not used as predictors for this reason.

## 2. Assessment Structure

Data quality is evaluated across five standard dimensions:

| Dimension | What we check |
|---|---|
| **Completeness** | Are all expected fields populated? |
| **Consistency** | Are values encoded uniformly across records? |
| **Validity** | Do values conform to expected formats and rules? |
| **Uniqueness** | Are there duplicate records? |
| **Accuracy** | Do values reflect plausible real-world facts? |

The sections below follow this structure. Each section documents the issue, quantifies its extent, and applies remediation where applicable.

## 3. Completeness

Completeness measures whether all expected fields are populated. We compute the missing count and percentage per column, then decide how to handle each case.

### 3.1 Missing Values per Column

We compute the missing count and missing percentage for every column, sorted by missingness.

In [100]:
missing_count = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(2)

missing_summary = (
    pd.DataFrame({'missing_count': missing_count, 'missing_%': missing_pct})
      .sort_values('missing_%', ascending=False)
)

missing_summary

,missing_count,missing_%
notes,500,99.60
financials.annual_salary,497,99.00
loan_purpose,452,90.04
processing_timestamp,440,87.65
decision.rejection_reason,292,58.17
decision.interest_rate,210,41.83
decision.approved_amount,210,41.83
applicant_info.ssn,5,1.00
applicant_info.ip_address,5,1.00
financials.annual_income,5,1.00


Columns with more than 95% missing values provide almost no analytical value and risk introducing noise. We drop *notes* (99.6% missing) immediately. *financials.annual_salary* (99% missing) will be merged with *financials.annual_income* in the next subsection before being dropped.

In [101]:
df_clean = df.copy()

# Drop the notes column (99.6% missing — no analytical value)
df_clean = df_clean.drop(columns=['notes'])

print('Shape before:', df.shape)
print('Shape after dropping notes:', df_clean.shape)

Shape before: (502, 21)
Shape after dropping notes: (502, 20)


### 3.2 annual_income vs annual_salary

The dataset contains two seemingly related fields: *financials.annual_income* and *financials.annual_salary*. We investigate whether they overlap or refer to different concepts.

In [102]:
# How many records have annual_income, annual_salary, or both?
has_income  = df['financials.annual_income'].notna()
has_salary  = df['financials.annual_salary'].notna()

print('Has annual_income only :', (has_income & ~has_salary).sum())
print('Has annual_salary only :', (~has_income & has_salary).sum())
print('Has both               :', (has_income & has_salary).sum())
print('Has neither            :', (~has_income & ~has_salary).sum())

Has annual_income only : 497
Has annual_salary only : 5
Has both               : 0
Has neither            : 0


The two fields are mutually exclusive — no record has both. They appear to be the same concept collected under different field names across different data entry points. We treat *financials.annual_salary* as an alias of *financials.annual_income* and merge the two columns into a single *financials.annual_income* column before dropping *financials.annual_salary*.

In [103]:
# Merge annual_salary into annual_income where income is missing
df_clean['financials.annual_income'] = df_clean['financials.annual_income'].fillna(df_clean['financials.annual_salary'])
print('annual_income missing after merge:', df_clean['financials.annual_income'].isna().sum())

# Drop annual_salary now that its values have been transferred
df_clean = df_clean.drop(columns=['financials.annual_salary'])
print('Shape after merging and dropping annual_salary:', df_clean.shape)

annual_income missing after merge: 0
Shape after merging and dropping annual_salary: (502, 19)


### 3.3 Spending Behavior Missingness

We verify that the *spending_behavior* field is fully populated, since it was described for all applicants in Section 1.

In [104]:
missing_spending = df_clean['spending_behavior'].isna().sum()
print(f'Missing spending_behavior: {missing_spending} ({missing_spending/len(df_clean)*100:.1f}%)')

Missing spending_behavior: 0 (0.0%)


Spending behavior has no missing values — every record contains at least one spending entry.

### 3.4 Findings

**Findings:**
- *notes* (99.6% missing) was dropped — it provides no analytical value.
- *financials.annual_salary* (99% missing) was found to be mutually exclusive with *financials.annual_income*; the two were merged and *annual_salary* dropped.
- After merging, *financials.annual_income* has 0 missing values — all 5 previously missing records were filled from *financials.annual_salary*.
- *spending_behavior* is fully populated (0 missing).
- Other notable missingness: *loan_purpose* (90%), *processing_timestamp* (88%), *decision.rejection_reason* (58%), *decision.approved_amount* and *decision.interest_rate* (42% each) — these reflect structural properties of the data (e.g., only approved loans have interest rates and approved amounts).

## 4. Consistency

Consistency checks whether values are encoded uniformly across records. We examine numeric types, categorical encodings, and date formats.

### 4.1 Numeric Data Types

After JSON loading, some numeric columns are stored as *object* (string) type. We enforce correct numeric types using *pd.to_numeric* with safe coercion.

In [105]:
df_clean.dtypes

_id                                  object
spending_behavior                    object
processing_timestamp                 object
applicant_info.full_name             object
applicant_info.email                 object
applicant_info.ssn                   object
applicant_info.ip_address            object
applicant_info.gender                object
applicant_info.date_of_birth         object
applicant_info.zip_code              object
financials.annual_income             object
financials.credit_history_months      int64
financials.debt_to_income           float64
financials.savings_balance            int64
decision.loan_approved                 bool
decision.rejection_reason            object
loan_purpose                         object
decision.interest_rate              float64
decision.approved_amount            float64
dtype: object

In [106]:
numeric_cols = [
    'financials.annual_income',
    'financials.debt_to_income',
    'financials.savings_balance',
    'financials.credit_history_months',
    'decision.approved_amount',
    'decision.interest_rate',
]

for c in numeric_cols:
    if c in df_clean.columns:
        before_na = df_clean[c].isna().sum()
        df_clean[c] = pd.to_numeric(df_clean[c], errors='coerce')
        after_na = df_clean[c].isna().sum()
        print(f'{c}: NA before={before_na}, NA after conversion={after_na}')

# Convert processing_timestamp to datetime
if 'processing_timestamp' in df_clean.columns:
    before_na = df_clean['processing_timestamp'].isna().sum()
    df_clean['processing_timestamp'] = pd.to_datetime(df_clean['processing_timestamp'], errors='coerce')
    after_na = df_clean['processing_timestamp'].isna().sum()
    print(f'processing_timestamp: NA before={before_na}, NA after conversion={after_na}')

financials.annual_income: NA before=0, NA after conversion=0
financials.debt_to_income: NA before=0, NA after conversion=0
financials.savings_balance: NA before=0, NA after conversion=0
financials.credit_history_months: NA before=0, NA after conversion=0
decision.approved_amount: NA before=210, NA after conversion=210
decision.interest_rate: NA before=210, NA after conversion=210
processing_timestamp: NA before=440, NA after conversion=440


No new missing values were introduced by numeric coercion, which confirms that all values were already in numeric-compatible format.

### 4.2 Gender Categorical Consistency

The *applicant_info.gender* field contains multiple encodings of the same categories: *"Female"*/*"Male"* and *"F"*/*"M"*. We standardise all values to a canonical set and set unmappable entries to missing.

In [107]:
print('Before standardisation:')
print(df_clean['applicant_info.gender'].value_counts(dropna=False).to_string())

Before standardisation:
applicant_info.gender
Male      195
Female    193
F          58
M          53
            2
NaN         1


In [108]:
g = df_clean['applicant_info.gender'].astype('string').str.strip().str.lower()

gender_map = {
    'm': 'Male',
    'male': 'Male',
    'f': 'Female',
    'female': 'Female',
}

df_clean['applicant_info.gender'] = g.map(gender_map)

print('After standardisation:')
print(df_clean['applicant_info.gender'].value_counts(dropna=False).to_string())

After standardisation:
applicant_info.gender
Female    251
Male      248
NaN         3


**Findings:** Gender was encoded as four variants (*Male*, *Female*, *M*, *F*) plus two blank strings and one *NaN*. After standardisation, all records map to *Male* or *Female*, with 3 records set to *NaN* (blank entries and the original missing value were unmappable).

### 4.3 Date Format Consistency

Two date/timestamp fields are present: *applicant_info.date_of_birth* and *processing_timestamp*. We inspect the formats actually used and parse them consistently.

#### 4.3.1 date_of_birth formats

We first inspect a sample of raw values before parsing to identify the formats in use.

In [109]:
# Look at a sample of unique date_of_birth values before parsing
sample_dobs = df_clean['applicant_info.date_of_birth'].dropna().unique()[:30]
print('Sample date_of_birth values:')
for d in sorted(sample_dobs):
    print(' ', d)

Sample date_of_birth values:
  
  01/12/1978
  14/02/1982
  18/07/1979
  1961-11-04
  1962-08-26
  1970-10-01
  1976-04-19
  1978-02-16
  1981-10-27
  1983-04-25
  1985-01-03
  1989-06-13
  1989-08-01
  1989-10-07
  1989-10-10
  1989-10-24
  1990-05-04
  1990/07/26
  1991-10-11
  1992-03-31
  1992-06-22
  1994-03-25
  1997-02-25
  1997-03-23
  1997-12-09
  1999-05-21
  20/04/1979
  2001-03-09
  28/01/1990


In [110]:
dob = pd.to_datetime(df_clean['applicant_info.date_of_birth'], errors='coerce')

invalid_dob = dob.isna().sum()
print('Invalid/unparseable date_of_birth:', invalid_dob)

# Fixed reference date for reproducibility
ref_date = pd.Timestamp('2024-01-15')

df_clean['applicant_info.age'] = ((ref_date - dob).dt.days // 365).astype('Int64')

# Remove impossible ages
out_of_range = ((df_clean['applicant_info.age'] < 0) | (df_clean['applicant_info.age'] > 120)).sum()
print('Out-of-range ages:', out_of_range)

df_clean.loc[(df_clean['applicant_info.age'] < 0) | (df_clean['applicant_info.age'] > 120), 'applicant_info.age'] = pd.NA

Invalid/unparseable date_of_birth: 162
Out-of-range ages: 0


The 162 unparseable entries appear to use alternative formats (e.g., *DD/MM/YYYY* (e.g. *14/02/1982*) and *YYYY/MM/DD* (e.g. *1990/07/26*)) that cannot be resolved without additional context. Note that formats like *DD/MM/YYYY* are ambiguous when the day value is 12 or below, as they could be misread as *MM/DD/YYYY* — these records cannot be safely parsed without additional locale context. We set age to missing for those records.

#### 4.3.2 processing_timestamp

We examine the coverage and values of *processing_timestamp* after datetime conversion.

In [111]:
print('processing_timestamp non-null count:', df_clean['processing_timestamp'].notna().sum())
print('Sample values:')
print(df_clean['processing_timestamp'].dropna().value_counts().head())

processing_timestamp non-null count: 62
Sample values:
processing_timestamp
2024-01-15 00:00:00+00:00    59
2025-12-01 00:00:00+00:00     1
2026-03-15 00:00:00+00:00     1
2027-01-20 00:00:00+00:00     1
Name: count, dtype: int64


Only ~12% of records have a timestamp. The majority share the value 2024-01-15, but 3 records contain future timestamps (2025-12-01, 2026-03-15, 2027-01-20) — which are impossible relative to the reference date and represent an additional data quality concern for audit trail purposes.

#### 4.3.3 Derived Feature: Age

We summarise the computed *age* column, which is derived from *date_of_birth* using a fixed reference date of 2024-01-15.

In [112]:
print('Age distribution summary:')
print(df_clean['applicant_info.age'].describe().to_string())
n_missing_age = df_clean['applicant_info.age'].isna().sum()
print(f'\nMissing ages (unparseable DOB): {n_missing_age}')

Age distribution summary:
count        340.0
mean     38.935294
std      11.149901
min           21.0
25%          30.75
50%           37.0
75%           45.0
max           65.0

Missing ages (unparseable DOB): 162


### 4.4 Findings

**Findings:**
- *financials.annual_income* was stored as *object* and converted to *float64*; no new missing values were introduced.
- Gender had four encodings (*Male*, *Female*, *M*, *F*) — standardised to two canonical labels; 3 records set to *NaN*.
- *date_of_birth* contains 162 unparseable values (likely alternative date formats); age set to *NaN* for those records.
- *processing_timestamp* is populated in only ~12% of records; most populated values share a single date (2024-01-15), but 3 records contain future timestamps, indicating both a batch-labelling issue and possible data entry errors.
- Computed *age* ranges from 21 to 65 years with a mean of ~39 years (for parseable records).

## 5. Validity

Validity checks whether field values conform to expected formats and rules. We check for logically impossible numeric values and format compliance for structured identifiers.

### 5.1 Negative Numeric Values

We detect values that are logically impossible — such as negative credit history months or negative income — and set them to missing.

In [113]:
invalid_credit_history = (df_clean['financials.credit_history_months'] < 0).sum()
invalid_dti = ((df_clean['financials.debt_to_income'] < 0) | (df_clean['financials.debt_to_income'] > 1)).sum()

# Negative annual_income
neg_income = (df_clean['financials.annual_income'] < 0).sum()
print('Negative annual_income:', neg_income)

# Negative interest_rate
neg_rate = (df_clean['decision.interest_rate'] < 0).sum()
print('Negative interest_rate:', neg_rate)

print('Invalid credit_history_months (<0):', invalid_credit_history)
print('Invalid debt_to_income (outside [0,1]):', invalid_dti)

Negative annual_income: 0
Negative interest_rate: 0
Invalid credit_history_months (<0): 2
Invalid debt_to_income (outside [0,1]): 1


In [114]:
df_clean.loc[df_clean['financials.credit_history_months'] < 0, 'financials.credit_history_months'] = pd.NA
df_clean.loc[
    (df_clean['financials.debt_to_income'] < 0) | (df_clean['financials.debt_to_income'] > 1),
    'financials.debt_to_income'
] = pd.NA
df_clean.loc[df_clean['financials.annual_income'] < 0, 'financials.annual_income'] = pd.NA
df_clean.loc[df_clean['decision.interest_rate'] < 0, 'decision.interest_rate'] = pd.NA

# Re-check
print('After fix:')
print('  Invalid credit_history_months:', (df_clean['financials.credit_history_months'] < 0).sum())
print('  Invalid debt_to_income:', ((df_clean['financials.debt_to_income'] < 0) | (df_clean['financials.debt_to_income'] > 1)).sum())
print('  Negative annual_income:', (df_clean['financials.annual_income'] < 0).sum())
print('  Negative interest_rate:', (df_clean['decision.interest_rate'] < 0).sum())

After fix:
  Invalid credit_history_months: 0
  Invalid debt_to_income: 0
  Negative annual_income: 0
  Negative interest_rate: 0


### 5.2 SSN Format

US Social Security Numbers should follow the pattern *NNN-NN-NNNN* (three digits, dash, two digits, dash, four digits).

In [115]:
ssn_pattern = re.compile(r'^\d{3}-\d{2}-\d{4}$')
invalid_ssn = df_clean['applicant_info.ssn'].dropna().apply(lambda x: not bool(ssn_pattern.match(str(x)))).sum()
total_ssn = df_clean['applicant_info.ssn'].notna().sum()
print(f'Invalid SSN format: {invalid_ssn} out of {total_ssn} non-null records')

Invalid SSN format: 0 out of 497 non-null records


### 5.3 ZIP Code Format

US ZIP codes should be 5-digit numeric strings.

In [116]:
zip_pattern = re.compile(r'^\d{5}$')
invalid_zip = df_clean['applicant_info.zip_code'].dropna().apply(lambda x: not bool(zip_pattern.match(str(x).strip()))).sum()
total_zip = df_clean['applicant_info.zip_code'].notna().sum()
print(f'Invalid ZIP code format: {invalid_zip} out of {total_zip} non-null records')

Invalid ZIP code format: 1 out of 501 non-null records


### 5.4 Email Format

We check that emails contain an *@* sign and a domain suffix (basic validation).

In [117]:
email_pattern = re.compile(r'^[^@]+@[^@]+\.[^@]+$')
invalid_email = df_clean['applicant_info.email'].dropna().apply(lambda x: not bool(email_pattern.match(str(x)))).sum()
total_email = df_clean['applicant_info.email'].notna().sum()
print(f'Invalid email format: {invalid_email} out of {total_email} non-null records')

Invalid email format: 10 out of 502 non-null records


### 5.5 Findings

**Findings:**
- Negative *credit_history_months*: 2 records — set to *NaN*.
- Debt-to-income ratio outside [0, 1]: 1 record — set to *NaN*.
- Negative *annual_income* and *interest_rate*: 0 records found (no action needed).
- SSN format: all non-null records conform to the *NNN-NN-NNNN* pattern.
- ZIP code format: 1 invalid record found — documented.
- Email format: 10 invalid records found — documented.

## 6. Uniqueness

We check for exact duplicate rows. Because *spending_behavior* contains nested structures (lists/dicts) that are not hashable, duplicate detection is performed on all other columns.

In [118]:
# Exclude columns that contain unhashable objects (e.g., lists/dicts like spending_behavior)
dup_subset = [c for c in df_clean.columns if c != 'spending_behavior']

dup_count = df_clean.duplicated(subset=dup_subset).sum()
dup_pct = (dup_count / len(df_clean)) * 100

print('Duplicate rows (excluding spending_behavior):', dup_count)
print('Duplicate rows (%):', round(dup_pct, 2))

Duplicate rows (excluding spending_behavior): 1
Duplicate rows (%): 0.2


In [119]:
df_clean = df_clean.drop_duplicates(subset=dup_subset).reset_index(drop=True)
print('Shape after dropping duplicates:', df_clean.shape)

Shape after dropping duplicates: (501, 20)


**Findings:**
- 1 exact duplicate row detected (0.2% of records) — removed.
- Final dataset after deduplication: 501 records.

## 7. Accuracy

Accuracy checks whether field values reflect plausible real-world facts, even when they are not technically invalid. Two checks are performed here.

### 7.1 Plausible Ages

We verify that the derived *age* values fall within a realistic range for credit applicants. Applicants under 18 cannot legally apply for credit in most jurisdictions.

In [120]:
# Age distribution check
print('Age distribution summary:')
print(df_clean['applicant_info.age'].describe().to_string())

# Ages below 18 (legally cannot apply for credit in most jurisdictions)
underage = (df_clean['applicant_info.age'] < 18).sum()
print(f'\nApplicants under 18: {underage}')

Age distribution summary:
count        339.0
mean     38.952802
std      11.161701
min           21.0
25%           30.5
50%           37.0
75%           45.0
max           65.0

Applicants under 18: 0


All ages are within a plausible adult range. No underage applicants detected.

### 7.2 Approved Loans with Zero or Missing Income

An approved loan with no reported income is a potential accuracy concern — it may reflect a data entry error or missing information rather than a genuine zero-income applicant.

In [121]:
# Approved loans where income is 0 or missing
approved = df_clean['decision.loan_approved'] == True
zero_income = (df_clean['financials.annual_income'] == 0) | (df_clean['financials.annual_income'].isna())

suspicious = (approved & zero_income).sum()
print(f'Approved loans with zero or missing income: {suspicious}')
if suspicious > 0:
    print(df_clean[approved & zero_income][['_id', 'financials.annual_income', 'decision.approved_amount']].to_string())

Approved loans with zero or missing income: 0


No approved loans with zero or missing income were detected — this accuracy check passed.

### 7.3 Findings

**Findings:**
- All computed ages fall between 21 and 65 years — no underage applicants detected.
- No approved loans with zero or missing income were detected.

## 8. Summary of Data Quality Issues

The table below summarises all issues identified and the remediation applied across the five quality dimensions.

In [122]:
summary_data = {
    'Dimension': ['Completeness', 'Completeness', 'Completeness', 'Consistency', 'Consistency', 'Consistency', 'Validity', 'Validity', 'Validity', 'Validity', 'Validity', 'Uniqueness', 'Accuracy'],
    'Issue': [
        '`notes` column (99.6% missing)',
        '`financials.annual_salary` merged into `annual_income` (99% missing separately)',
        'financials.annual_income: 0 missing after salary merge',
        'Gender encoded as M/F and Male/Female — standardised',
        '162 date_of_birth records could not be parsed — age set to missing',
        '`processing_timestamp` present in only ~12% of records',
        'Negative `credit_history_months`: 2 records',
        'Debt-to-income ratio outside [0,1]: 1 record',
        'SSN format violations: 0 records',
        'ZIP code format violations: 1 record',
        'Email format violations: 10 records',
        'Duplicate rows: 1 (0.2%)',
        'Approved loans with zero or missing income: 0 found'
    ],
    'Records affected': ['500', '497', '0', '111', '162', '440', '2', '1', 'see above', '1', '10', '1', '0'],
    'Action taken': [
        'Dropped',
        'Merged then dropped',
        'Left as NaN',
        'Mapped to Male/Female; 3 unmappable → NaN',
        'Age set to NaN for affected records',
        'Converted to datetime; not dropped',
        'Set to NaN',
        'Set to NaN',
        'Documented',
        'Documented',
        'Documented',
        'Removed',
        'None — check passed'
    ]
}

summary_df = pd.DataFrame(summary_data)
summary_df

,Dimension,Issue,Records affected,Action taken
0,Completeness,`notes` column (99.6% missing),500,Dropped
1,Completeness,`financials.annual_salary` merged into `annual...,497,Merged then dropped
2,Completeness,financials.annual_income: 0 missing after sala...,0,Left as NaN
3,Consistency,Gender encoded as M/F and Male/Female — standa...,111,Mapped to Male/Female; 3 unmappable → NaN
4,Consistency,162 date_of_birth records could not be parsed ...,162,Age set to NaN for affected records
5,Consistency,`processing_timestamp` present in only ~12% of...,440,Converted to datetime; not dropped
6,Validity,Negative `credit_history_months`: 2 records,2,Set to NaN
7,Validity,"Debt-to-income ratio outside [0,1]: 1 record",1,Set to NaN
8,Validity,SSN format violations: 0 records,see above,Documented
9,Validity,ZIP code format violations: 1 record,1,Documented


After applying all remediation steps, the cleaned dataset contains 501 records and is saved to *data/credit_applications_clean.csv* for use in the bias analysis notebook.

In [123]:
df_clean.to_csv(
    '../data/credit_applications_clean.csv',
    index=False
)